Dataset loading

Loading the CelebA attributes file using pandas. 
CelebA contains 202,599 face images with 40 binary attributes each (such as Smiling, Male, Young, Attractive, etc.).
Values are encoded as 1 (attribute present) and -1 (attribute absent).
Skipping the first row since it contains only the total image count and using image filename as the index.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv('data/list_attr_celeba.txt', skiprows=1, sep='\s+', index_col=0)

df.head()

,5_o_Clock_Shadow,Arched_Eyebrows,Attractive,Bags_Under_Eyes,Bald,Bangs,Big_Lips,Big_Nose,Black_Hair,Blond_Hair,...,Sideburns,Smiling,Straight_Hair,Wavy_Hair,Wearing_Earrings,Wearing_Hat,Wearing_Lipstick,Wearing_Necklace,Wearing_Necktie,Young
000001.jpg,-1,1,1,-1,-1,-1,-1,-1,-1,-1,...,-1,1,1,-1,1,-1,1,-1,-1,1
000002.jpg,-1,-1,-1,1,-1,-1,-1,1,-1,-1,...,-1,1,-1,-1,-1,-1,-1,-1,-1,1
000003.jpg,-1,-1,-1,-1,-1,-1,1,-1,-1,-1,...,-1,-1,-1,1,-1,-1,-1,-1,-1,1
000004.jpg,-1,-1,1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,1,-1,1,-1,1,1,-1,1
000005.jpg,-1,1,1,-1,-1,-1,1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,1,-1,-1,1


Dataset overview

Checking the basic structure of the dataset: 
1. the number of rows and columns
2. the full list of attributes
3. missing values

In [2]:
print("Dataset size:", df.shape)

print("\nAll attributes:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sum())

Dataset size: (202599, 40)

All attributes:
['5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young']

Missing values:
0


Data cleanup

Dataset uses 1 and -1 as binary values, so converting them to 1 and 0 for easier analysis and visualization. 
Verifying that all columns have the correct data type (integer).

In [4]:
df_clean = df.replace(-1, 0)

print("Data types:")
print(df_clean.dtypes.value_counts())

print("\nUnique values in dataset:")
print(df_clean.stack().unique())

df_clean.head()

Data types:
int64    40
Name: count, dtype: int64

Unique values in dataset:
[0 1]


,5_o_Clock_Shadow,Arched_Eyebrows,Attractive,Bags_Under_Eyes,Bald,Bangs,Big_Lips,Big_Nose,Black_Hair,Blond_Hair,...,Sideburns,Smiling,Straight_Hair,Wavy_Hair,Wearing_Earrings,Wearing_Hat,Wearing_Lipstick,Wearing_Necklace,Wearing_Necktie,Young
000001.jpg,0,1,1,0,0,0,0,0,0,0,...,0,1,1,0,1,0,1,0,0,1
000002.jpg,0,0,0,1,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,1
000003.jpg,0,0,0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,0,1
000004.jpg,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,1,0,1,1,0,1
000005.jpg,0,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,1


Data transformation

Create 4 new numerical columns based on existing binary attributes.
These columns will be used further.

In [7]:
original_cols = df.columns.tolist()
df_clean['attr_sum'] = df_clean[original_cols].sum(axis=1)

# Makeup score
makeup_cols = ['Heavy_Makeup', 'Wearing_Lipstick', 'Rosy_Cheeks', 
               'Wearing_Earrings', 'Wearing_Necklace']
df_clean['makeup_score'] = df_clean[makeup_cols].sum(axis=1)

# Femininity score
feminine_cols = ['Arched_Eyebrows', 'Heavy_Makeup', 'High_Cheekbones', 
                 'Wearing_Lipstick', 'Wavy_Hair', 'Bangs']
df_clean['femininity_score'] = df_clean[feminine_cols].sum(axis=1)

# Has hair
hair_cols = ['Black_Hair', 'Blond_Hair', 'Brown_Hair', 'Gray_Hair']
df_clean['has_hair'] = (df_clean[hair_cols].sum(axis=1) > 0).astype(int)

print("New columns added:")
print(df_clean[['attr_sum', 'makeup_score', 'femininity_score', 'has_hair']].head(10))

New columns added!
            attr_sum  makeup_score  femininity_score  has_hair
000001.jpg        13             3                 4         1
000002.jpg         8             0                 1         1
000003.jpg         8             0                 1         0
000004.jpg         8             3                 1         0
000005.jpg         9             2                 3         0
000006.jpg        11             3                 4         1
000007.jpg        12             0                 0         1
000008.jpg         9             0                 1         1
000009.jpg        15             4                 5         0
000010.jpg         7             2                 4         0


Descriptive Statistics

Computation of basic statistics for new 4 numerical columns: 
attr_sum, makeup_score, femininity_score, and has_hair.
This gives an overview of the distribution of these values across the dataset.

In [8]:
stats_cols = ['attr_sum', 'makeup_score', 'femininity_score', 'has_hair']

df_clean[stats_cols].describe().round(2)

,attr_sum,makeup_score,femininity_score,has_hair
count,202599.00,202599.00,202599.00,202599.00
mean,9.03,1.24,2.05,0.62
std,2.94,1.37,1.73,0.49
min,1.00,0.00,0.00,0.00
25%,7.00,0.00,0.00,0.00
50%,9.00,1.00,2.00,1.00
75%,11.00,2.00,4.00,1.00
max,20.00,5.00,6.00,1.00
